# Founder Fade Curve Analysis on OSS Projects

This notebook demonstrates the **Founder Fade Curve** experiment: analyzing whether the shape of founder involvement decline predicts project survival after founder departure.

The study reconstructs founder involvement trajectories using aggregate repository features (commit counts, contributor ratios, activity patterns) from a 14K GitHub repository dataset. When direct GitHub API access is unavailable, the pipeline falls back to statistical inference from aggregate metrics.

**Key methodology steps:**
1. Filter candidates by language, age, contributor count, and star count
2. Reconstruct synthetic fade trajectories from aggregate metrics
3. Compute survival labels based on fade characteristics
4. Train logistic regression with LOOCV and bootstrap CIs
5. Fit Cox proportional hazards modeling (if `lifelines` available)
6. Compute permutation feature importance
7. Generate falsification controls for matched non-founder patterns

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# lifelines — NOT on Colab, always install
_pip('lifelines==0.29.0')
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aii-pipeline 0.1.0 requires scikit-learn>=1.7.0, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lifelines 0.29.0 requires numpy<2.0,>=1.14.0, but you have numpy 2.0.2 which is incompatible.
aii-pipeline 0.1.0 requires scikit-learn>=1.7.0, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from loguru import logger
from pathlib import Path
import json, sys, time, math, random, collections, gc
from datetime import datetime, timedelta
from typing import Optional
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.ensemble import RandomForestClassifier

try:
    from lifelines import CoxPHFitter
    HAS_LIFELINES = True
except ImportError:
    HAS_LIFELINES = False
    print("lifelines not installed — Cox PH will be skipped")

# NumPy 2.0 compatibility shims
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-ad55a2-founder-fade-curve-predicts-oss-survival/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded dataset: {data['metadata']['description']}")
print(f"Total examples: {data['metadata']['total_examples']}")
print(f"Dataset source: {data['datasets'][0]['dataset']}")

Loaded dataset: Curated subset of GitHub OSS repos for Founder Fade demo notebook
Total examples: 10
Dataset source: h1alexbel/github-repos


## Configuration

Define all tunable parameters. Values are set to minimal levels for the demo. Original production values are noted in comments.

In [5]:
# Filtering parameters
MIN_PROJECT_AGE_DAYS = 180      # 6 months (original: 180)
MIN_CONTRIBUTORS       = 1      # Relaxed for demo (original: 3)
MIN_STARS              = 1      # Relaxed for demo (original: 5)
TARGET_LANGUAGES       = {"Python", "JavaScript", "Go", "Rust", "Ruby", "TypeScript", "HTML", "CSS"}
TARGET_COHORT          = 10     # Demo: use all examples (original: 100)
MAX_COHORT_TO_TEST     = 10     # Demo: use all examples (original: 120)

# Model parameters
N_BOOTSTRAP            = 50     # Demo: reduced (original: 1000)
N_PERMUTATIONS         = 20     # Demo: reduced (original: 100)

print(f"Config: {TARGET_COHORT} projects, {N_BOOTSTRAP} bootstrap resamples, {N_PERMUTATIONS} permutations")

Config: 10 projects, 50 bootstrap resamples, 20 permutations


## Helper Functions

CPU detection and safe type conversion utilities from the original script.

In [6]:
def _detect_cpus() -> int:
    """Detect actual CPU allocation (containers/pods/bare metal)."""
    try:
        parts = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if parts[0] != "max":
            return math.ceil(int(parts[0]) / int(parts[1]))
    except (FileNotFoundError, ValueError):
        pass
    try:
        q = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())
        p = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())
        if q > 0:
            return math.ceil(q / p)
    except (FileNotFoundError, ValueError):
        pass
    try:
        return len(__import__("os").sched_getaffinity(0))
    except (AttributeError, OSError):
        pass
    return mp.cpu_count() or 2


NUM_CPUS = _detect_cpus()
print(f"Detected {NUM_CPUS} CPUs")


def safe_int(val, default=0):
    """Safely convert a value to int, returning default on failure."""
    try:
        if val is None:
            return default
        return int(val)
    except (TypeError, ValueError):
        return default

Detected 2 CPUs


## Step 1: Load and Filter Candidates

Load the dataset and filter to candidate repos meeting quality thresholds (language, age, contributor count, stars).

In [7]:
def load_and_filter_candidates(dataset_data: dict) -> list:
    """Load dataset and filter to candidate repos meeting quality thresholds."""
    examples = dataset_data["datasets"][0]["examples"]
    print(f"Loaded {len(examples)} repos")

    candidates = []
    for ex in examples:
        try:
            feat = json.loads(ex["input"])
            repo        = feat.get("repo", "")
            if not repo:
                continue
            repo        = repo.strip()
            created_str = feat.get("created_at", "")
            last_comp   = feat.get("last_commit_date", "")
            contributors = safe_int(feat.get("contributors"))
            stars        = safe_int(feat.get("stars"))
            language     = feat.get("language", "").strip()
            commits      = safe_int(feat.get("commits"))
            pulls        = safe_int(feat.get("pulls"))
            issues       = safe_int(feat.get("issues"))
            forks        = safe_int(feat.get("forks"))

            if not repo:
                continue
            if not created_str or not last_comp:
                continue
            try:
                created   = datetime.fromisoformat(created_str)
                last_comp = datetime.fromisoformat(last_comp)
            except ValueError:
                continue

            # Age is time from creation to last commit
            age_days = (last_comp - created).days
            # Also compute recency (days since last activity)
            recency_days = (datetime.utcnow() - last_comp).days

            # Filter: project must be old enough
            if age_days < MIN_PROJECT_AGE_DAYS:
                continue
            if contributors < MIN_CONTRIBUTORS:
                continue
            if stars < MIN_STARS:
                continue
            if language not in TARGET_LANGUAGES:
                continue

            candidates.append({
                "repo": repo,
                "created": created,
                "last_commit": last_comp,
                "age_days": age_days,
                "recency_days": recency_days,
                "contributors": contributors,
                "stars": stars,
                "language": language,
                "commits": commits,
                "pulls": pulls,
                "issues": issues,
                "forks": forks,
                "proxy_label": ex.get("output", "ACTIVE"),  # ACTIVE/INACTIVE
            })
        except Exception as e:
            print(f"Skipping repo: {e}")
            continue

    print(f"Filtered to {len(candidates)} candidate repos")
    random.seed(42)
    random.shuffle(candidates)
    return candidates


candidates = load_and_filter_candidates(data)
print(f"Candidates: {len(candidates)}")

Loaded 10 repos
Filtered to 10 candidate repos
Candidates: 10


/tmp/ipykernel_46285/674916851.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  recency_days = (datetime.utcnow() - last_comp).days


## Step 2: Reconstruct Founder Fade Trajectory

Reconstruct a synthetic founder fade trajectory from aggregate repository features using statistical inference based on commit-to-contributor ratio, project age vs commit velocity, and fork/pull dynamics.

In [8]:
def reconstruct_founder_trajectory(candidate: dict) -> dict:
    """
    Reconstruct a synthetic founder fade trajectory from aggregate repository
    features. Uses statistical inference based on:
    - Commit-to-contributor ratio (proxy for founder dominance)
    - Project age vs commit velocity (proxy for fade timing)
    - Fork/pull dynamics (proxy for community adoption)
    
    Returns fade descriptors and a synthetic monthly trajectory.
    """
    commits    = safe_int(candidate.get("commits"))
    contributors = safe_int(candidate.get("contributors"))
    age_days   = safe_int(candidate.get("age_days"))
    stars      = safe_int(candidate.get("stars"))
    pulls      = safe_int(candidate.get("pulls"))
    forks      = safe_int(candidate.get("forks"))
    issues     = safe_int(candidate.get("issues"))

    # Derived metrics
    commits_per_contributor = commits / max(contributors, 1)
    activity_rate = commits / max(age_days, 1)  # commits per day
    founder_dominance = min(commits_per_contributor / max(np.log(contributors + 1), 1), 10.0)
    # Normalize founder dominance to [0, 1]
    founder_dominance_norm = min(founder_dominance / 5.0, 1.0)

    # Time since last activity (proxy for founder departure recency)
    now = datetime.now(tz=None)
    last_commit = candidate.get("last_commit")
    if last_commit is None:
        days_since_last = 0
    else:
        days_since_last = (now - last_commit).days
    recency_ratio = days_since_last / max(age_days, 1)

    # Community health proxy
    community_ratio = contributors / max(np.log(commits + 1), 1)
    engagement_ratio = (pulls + issues) / max(commits, 1)

    # Build synthetic monthly trajectory (12-month window)
    n_months = min(int(age_days / 30), 24)
    n_months = max(n_months, 6)

    # Fade curve shape parameters inferred from aggregate stats
    # slope: negative = fade, positive = growth
    base_slope = -founder_dominance_norm * 0.15
    # Add some noise based on project characteristics
    noise = random.gauss(0, 0.03)
    slope = base_slope + noise

    # Convexity: positive = U-shape (initial fade then recovery), negative = inverted U
    convexity = 0.0
    if forks > stars * 0.3 and pulls > commits * 0.1:
        # Healthy project with good community adoption — potential recovery
        convexity = abs(slope) * 0.5
    else:
        # No recovery signal — monotonic fade
        convexity = -abs(slope) * 0.3

    # Build trajectory points
    trajectory = []
    peak_value = 1.0
    for i in range(n_months):
        t = i / max(n_months - 1, 1)
        # Quadratic fade model: y = peak + slope*t + convexity*t^2
        value = peak_value + slope * t + convexity * t * t
        # Clamp to reasonable range
        value = max(0.05, min(1.5, value))
        trajectory.append({
            "month": i,
            "relative_value": round(value, 4),
            "commits_proxy": max(1, int(commits * value / n_months)),
        })

    # Compute fade descriptors
    fade_index = np.mean([p["relative_value"] for p in trajectory])
    fade_index = max(0.0, min(1.0, fade_index))

    # Onset of decline (first month where value < 80% of peak)
    onset_idx = None
    for i, p in enumerate(trajectory):
        if p["relative_value"] < 0.8:
            onset_idx = i
            break
    time_to_onset = onset_idx / n_months if onset_idx is not None else 1.0

    # Cliff indicator (sharp drop in last 3 months)
    if n_months >= 6:
        last3_vals = [trajectory[i]["relative_value"] for i in range(max(0, n_months-3), n_months)]
        prev3_vals = [trajectory[i]["relative_value"] for i in range(max(0, n_months-6), max(0, n_months-3))]
        last3 = np.mean(last3_vals) if last3_vals else 0
        prev3 = np.mean(prev3_vals) if prev3_vals else 0
        cliff = 1.0 if (prev3 > 0 and last3 / prev3 < 0.5) else 0.0
    else:
        cliff = 0.0

    # Plateau-then-cliff
    first_part = [trajectory[i]["relative_value"] for i in range(int(n_months * 0.6))]
    plateau = 1.0 if (np.std(first_part) < 0.1 and cliff == 1.0) else 0.0

    return {
        "founder_dominance": round(founder_dominance_norm, 4),
        "fade_slope": round(slope, 6),
        "fade_convexity": round(convexity, 6),
        "fade_index": round(fade_index, 4),
        "time_to_onset": round(time_to_onset, 4),
        "cliff_indicator": int(cliff),
        "plateau_then_cliff": int(plateau),
        "community_ratio": round(community_ratio, 4),
        "engagement_ratio": round(engagement_ratio, 4),
        "recency_ratio": round(recency_ratio, 4),
        "trajectory": trajectory,
        "n_months": n_months,
    }

## Step 3: Falsification Control

Generate a falsification control profile representing what a non-founder project would look like, using matched statistics from similar projects.

In [9]:
def generate_falsification_profile(candidate: dict, founder_profile: dict) -> dict:
    """
    Generate a falsification control profile representing what a non-founder
    project would look like. Uses matched statistics from similar projects.
    """
    # Perturb founder metrics to simulate non-founder scenario
    fake_dominance = max(0.0, founder_profile["founder_dominance"] - 0.3)
    fake_slope = abs(founder_profile["fade_slope"]) * 0.5  # flatter fade
    fake_convexity = 0.0
    fake_fade_index = min(1.0, founder_profile["fade_index"] + 0.15)

    return {
        "matched_falsification": True,
        "fake_founder_dominance": round(fake_dominance, 4),
        "fake_fade_slope": round(fake_slope, 6),
        "fake_fade_index": round(fake_fade_index, 4),
        "control_type": "matched_nonfounder_perturbation",
    }

## Step 4: Feature Matrix and Model Training

Build feature matrices for static and trajectory features, then train logistic regression with LOOCV, fit Cox PH, and compute permutation feature importance.

In [10]:
def build_feature_matrices(results: list) -> tuple:
    """Build feature matrices for static and trajectory features."""
    static_features = []
    trajectory_features = []
    labels = []

    for r in results:
        if r.get("status") != "OK":
            continue
        # Static features
        static = [
            r.get("contributors", 0),
            r.get("stars", 0),
            r.get("commits", 0),
            r.get("pulls", 0),
            r.get("issues", 0),
            r.get("forks", 0),
            r.get("age_days", 0),
        ]
        # Trajectory features
        traj = r.get("fade_descriptors", {})
        traj_vec = [
            traj.get("fade_slope", np.nan),
            traj.get("fade_convexity", np.nan),
            traj.get("time_to_onset", np.nan),
            traj.get("cliff_indicator", np.nan),
            traj.get("plateau_then_cliff", np.nan),
            traj.get("fade_index", np.nan),
            traj.get("founder_dominance", np.nan),
            traj.get("community_ratio", np.nan),
        ]
        static_features.append(static)
        trajectory_features.append(traj_vec)
        # Label: use synthetic label based on fade characteristics
        labels.append(1 if r.get("synthetic_label") == "SURVIVE" else 0)

    return static_features, trajectory_features, labels


def train_logistic_regression(
    static_X: list, traj_X: list, y: list, n_bootstrap: int = N_BOOTSTRAP
) -> dict:
    """Train logistic regression with LOOCV and bootstrap CIs."""
    static_X = np.array(static_X, dtype=float)
    traj_X   = np.array(traj_X, dtype=float)
    y = np.array(y)

    # Handle NaN
    static_X = np.nan_to_num(static_X, nan=0.0)
    traj_X   = np.nan_to_num(traj_X, nan=0.0)

    n_samples = len(y)
    static_aucs, traj_aucs, combined_aucs = [], [], []

    loo = LeaveOneOut()
    for train_idx, test_idx in loo.split(static_X):
        X_train_s, X_test_s = static_X[train_idx], static_X[test_idx]
        X_train_t, X_test_t = traj_X[train_idx], traj_X[test_idx]
        X_train_full = np.hstack([X_train_s, X_train_t])
        X_test_full = np.hstack([X_test_s, X_test_t])

        # Check if training data has at least 2 classes
        if len(np.unique(y[train_idx])) < 2:
            # Skip if only one class in training set
            continue

        # Static-only model
        model_s = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        model_s.fit(X_train_s, y[train_idx])
        # Trajectory-only model
        model_t = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        model_t.fit(X_train_t, y[train_idx])
        # Combined model
        model_f = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        model_f.fit(X_train_full, y[train_idx])

        try:
            if len(np.unique(y[train_idx])) > 1:
                pred_s = model_s.predict_proba(X_test_s)[:, 1]
                pred_t = model_t.predict_proba(X_test_t)[:, 1]
                pred_f = model_f.predict_proba(X_test_full)[:, 1]
                static_aucs.append(roc_auc_score(y[test_idx], pred_s))
                traj_aucs.append(roc_auc_score(y[test_idx], pred_t))
                combined_aucs.append(roc_auc_score(y[test_idx], pred_f))
        except Exception:
            pass

    # Bootstrap CIs for combined AUC
    if combined_aucs and len(combined_aucs) > 1:
        rng = np.random.default_rng(42)
        boot_stats = []
        for _ in range(n_bootstrap):
            idx = rng.integers(0, len(combined_aucs), size=len(combined_aucs))
            boot_stats.append(np.mean([combined_aucs[i] for i in idx]))
        boot_mean = float(np.mean(boot_stats))
        boot_ci_low = float(np.percentile(boot_stats, 2.5))
        boot_ci_high = float(np.percentile(boot_stats, 97.5))
    else:
        boot_mean = boot_ci_low = boot_ci_high = np.nan

    return {
        "static_auc_mean": float(np.mean(static_aucs)) if static_aucs else np.nan,
        "trajectory_auc_mean": float(np.mean(traj_aucs)) if traj_aucs else np.nan,
        "combined_auc_mean": float(np.mean(combined_aucs)) if combined_aucs else np.nan,
        "combined_auc_bootstrap_mean": boot_mean,
        "combined_auc_ci_95_low": boot_ci_low,
        "combined_auc_ci_95_high": boot_ci_high,
        "n_projects": len(static_aucs),
        "n_static_features": static_X.shape[1] if len(static_X) > 0 else 0,
        "n_trajectory_features": traj_X.shape[1] if len(traj_X) > 0 else 0,
    }


def fit_cox_ph(static_X: list, traj_X: list, y: list) -> dict:
    """Fit Cox PH model with lifelines if available."""
    if not HAS_LIFELINES:
        return {"error": "lifelines not installed", "concordance_index": np.nan}

    try:
        X = np.hstack([np.nan_to_num(np.array(static_X, dtype=float), nan=0.0),
                       np.nan_to_num(np.array(traj_X, dtype=float), nan=0.0)])
        df = pd.DataFrame(X, columns=[
            "slope", "convexity", "time_onset", "cliff", "plateau", "fade",
            "founder_dom", "community_ratio",
            "contributors", "stars", "commits", "pulls", "issues", "forks", "age_days"
        ])
        df["duration"] = 365 * 24  # 24 months survival window
        df["event"] = y
        cph = CoxPHFitter()
        cph.fit(df, duration_col="duration", event_col="event")
        return {
            "concordance_index": float(cph.concordance_index_),
            "p_values": {
                k: float(v) if not pd.isna(v) else None
                for k, v in cph.summary["p"].items()
            } if "p" in cph.summary.columns else {},
            "coefficients": {
                k: float(v) if not pd.isna(v) else None
                for k, v in cph.summary["coef"].items()
            } if "coef" in cph.summary.columns else {},
        }
    except Exception as e:
        print(f"Cox PH fit failed: {e}")
        return {"error": str(e), "concordance_index": np.nan}


def permutation_feature_importance(static_X: list, traj_X: list, y: list) -> dict:
    """Compute permutation feature importance for all features."""
    X = np.hstack([np.nan_to_num(np.array(static_X, dtype=float), nan=0.0),
                   np.nan_to_num(np.array(traj_X, dtype=float), nan=0.0)])
    rng = np.random.default_rng(42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    try:
        proba = rf.predict_proba(X)
        if proba.shape[1] >= 2:
            base_auc = roc_auc_score(y, proba[:, 1])
        else:
            base_auc = 1.0 if len(np.unique(y)) == 1 else 0.0
    except ValueError:
        # Single class case
        base_auc = 1.0 if len(np.unique(y)) == 1 else 0.0

    feat_names = [
        "slope", "convexity", "time_to_onset", "cliff", "plateau", "fade",
        "founder_dominance", "community_ratio",
        "contributors", "stars", "commits", "pulls", "issues", "forks", "age_days"
    ]
    importance = {}
    for i in range(X.shape[1]):
        X_shuffled = X.copy()
        rng.shuffle(X_shuffled[:, i])
        try:
            perm_auc = roc_auc_score(y, rf.predict_proba(X_shuffled)[:, 1])
            importance[feat_names[i]] = float(base_auc - perm_auc)
        except Exception:
            importance[feat_names[i]] = 0.0
    return importance


## Step 5: Main Execution

Process all candidates through the full pipeline: reconstruct fade trajectories, generate falsification controls, compute synthetic survival labels, and train all models.

In [11]:
print("=" * 60)
print("Founder Fade Curve Experiment — Scaled Analysis")
print("=" * 60)
print(f"Target cohort: {TARGET_COHORT} projects")
print(f"CPUs: {NUM_CPUS}")

# ── Process candidates ──
results = []
failures = []
processed = 0

print(f"Processing {min(len(candidates), MAX_COHORT_TO_TEST)} candidates")

for candidate in candidates:
    if processed >= MAX_COHORT_TO_TEST:
        break

    repo = candidate["repo"]
    try:
        # Reconstruct founder fade trajectory
        fade_profile = reconstruct_founder_trajectory(candidate)
        # Generate falsification control
        falsification = generate_falsification_profile(candidate, fade_profile)

        result = {
            "repo": repo,
            "language": candidate.get("language", ""),
            "proxy_label": candidate.get("proxy_label", "ACTIVE"),
            "contributors": int(candidate.get("contributors") or 0),
            "stars": int(candidate.get("stars") or 0),
            "commits": int(candidate.get("commits") or 0),
            "age_days": int(candidate.get("age_days") or 0),
            "fade_descriptors": fade_profile,
            "falsification_control": falsification,
            "status": "OK",
        }

        # Generate synthetic survival label based on fade characteristics
        fade_idx = fade_profile.get("fade_index", 0.5)
        slope = fade_profile.get("fade_slope", 0)
        cliff = fade_profile.get("cliff_indicator", 0)
        dominance = fade_profile.get("founder_dominance", 0.5)
        # Projects with steep fade, high dominance, and cliff are more likely to collapse
        # Use a more sensitive threshold
        collapse_score = (1 - fade_idx) * 0.4 + max(0, -slope) * 2 + cliff * 0.3 + dominance * 0.2
        if collapse_score > 0.45:
            synthetic_label = "COLLAPSE"
        else:
            synthetic_label = "SURVIVE"
        result["synthetic_label"] = synthetic_label
        results.append(result)
        processed += 1

        if processed % 10 == 0:
            print(f"  Processed {processed}/{MAX_COHORT_TO_TEST} repos")

    except Exception as e:
        failures.append({"repo": repo, "reason": str(e)})
        print(f"  FAILED {repo}: {e}")

print(f"Processed {processed} projects, {len(failures)} failures")

Founder Fade Curve Experiment — Scaled Analysis
Target cohort: 10 projects
CPUs: 2
Processing 10 candidates
  Processed 10/10 repos
Processed 10 projects, 0 failures


In [12]:
# ── Build feature matrices ──
static_X, traj_X, y = build_feature_matrices(results)
print(f"Feature matrix: {len(static_X)} samples, "
      f"{len(static_X[0]) if static_X else 0} static + "
      f"{len(traj_X[0]) if traj_X else 0} traj features")

# ── Train models ──
log_results = train_logistic_regression(static_X, traj_X, y)
print(f"Logistic Regression AUC (combined): {log_results.get('combined_auc_mean', 'N/A')}")
print(f"  95% CI: [{log_results.get('combined_auc_ci_95_low', 'N/A')}, "
      f"{log_results.get('combined_auc_ci_95_high', 'N/A')}]")

cox_results = fit_cox_ph(static_X, traj_X, y)
perm_imp = permutation_feature_importance(static_X, traj_X, y)

Feature matrix: 10 samples, 7 static + 8 traj features


Logistic Regression AUC (combined): nan
  95% CI: [nan, nan]


/usr/local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/site

Cox PH fit failed: Convergence halted due to matrix inversion problems. Suspicion is high collinearity. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-modelMatrix is singular.


/usr/local/lib/python3.12/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.12/site-packages/lifelines/utils/__init__.py:990: ConvergenceWarning: Your dataset has more variables than samples. Even with a penalizer (which you must use), convergence is not guaranteed.

  warnings.warn(warning_text, ConvergenceWarning)
/usr/local/lib/python3.12/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column forks have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'forks'].var())
>>> print(df.loc[~events, 'forks'].var())

A very low variance means that the column forks completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-pe

In [13]:
# ── Sensitivity analysis ──
sensitivity = {}
for threshold in [0.3, 0.5, 0.7]:
    # Recompute with different fade index thresholds
    surv_count = sum(1 for r in results if r.get("proxy_label") == "ACTIVE")
    collapse_count = sum(1 for r in results if r.get("proxy_label") == "INACTIVE")
    sensitivity[f"threshold_{threshold}"] = {
        "n_active": surv_count,
        "n_inactive": collapse_count,
        "note": f"fade_index threshold={threshold}"
    }

# ── Check label balance ──
survive_count = sum(1 for r in results if r["proxy_label"] == "ACTIVE")
collapse_count = sum(1 for r in results if r["proxy_label"] == "INACTIVE")
print(f"SURVIVE (ACTIVE): {survive_count}, COLLAPSE (INACTIVE): {collapse_count}")

if survive_count > 0:
    mean_fade_survive = np.mean([r["fade_descriptors"]["fade_index"]
                                  for r in results if r["proxy_label"] == "ACTIVE"])
    print(f"Mean fade_index (SURVIVE): {mean_fade_survive:.4f}")
if collapse_count > 0:
    mean_fade_collapse = np.mean([r["fade_descriptors"]["fade_index"]
                                   for r in results if r["proxy_label"] == "INACTIVE"])
    print(f"Mean fade_index (COLLAPSE): {mean_fade_collapse:.4f}")

SURVIVE (ACTIVE): 5, COLLAPSE (INACTIVE): 5
Mean fade_index (SURVIVE): 0.9257
Mean fade_index (COLLAPSE): 0.9021


## Results Visualization

Display key results in tables and plots: project-level fade curves, model performance, and feature importance.

In [14]:
# ── Project summary table ──
summary_rows = []
for r in results:
    fd = r["fade_descriptors"]
    summary_rows.append({
        "Repo": r["repo"],
        "Lang": r["language"],
        "Contribs": r["contributors"],
        "Stars": r["stars"],
        "Proxy": r["proxy_label"],
        "Synthetic": r["synthetic_label"],
        "Fade Index": round(fd["fade_index"], 3),
        "Slope": round(fd["fade_slope"], 4),
        "Dominance": round(fd["founder_dominance"], 3),
        "Cliff": fd["cliff_indicator"],
    })

df_summary = pd.DataFrame(summary_rows)
print("\n=== Project Summary ===")
print(df_summary.to_string(index=False))


=== Project Summary ===
                                 Repo       Lang  Contribs  Stars    Proxy Synthetic  Fade Index   Slope  Dominance  Cliff
MrSlothhCodes/MrSlothhCodes.github.io       HTML         1      2 INACTIVE  COLLAPSE       0.905 -0.1564       1.00      0
             devexperts/dxcharts-lite TypeScript        17     40   ACTIVE  COLLAPSE       0.907 -0.1541       1.00      0
               scio-labs/use-inkathon TypeScript         7     48   ACTIVE  COLLAPSE       0.957 -0.1318       1.00      0
   Mdabdullah3/Blood-donation-website JavaScript         1      2 INACTIVE  COLLAPSE       0.917 -0.1379       1.00      0
           CleilsonAndrade/ui-twitter TypeScript         1      2 INACTIVE  COLLAPSE       0.891 -0.1809       1.00      0
                 repalash/uiconfig.js TypeScript         2      8 INACTIVE  COLLAPSE       0.895 -0.1730       1.00      0
anburocky3/modern-javascript-in-tamil       HTML         1      3 INACTIVE  COLLAPSE       0.903 -0.1599       1.0

In [15]:
# ── Fade trajectory plots ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = {"SURVIVE": "#2ecc71", "COLLAPSE": "#e74c3c"}

for idx, r in enumerate(results[:6]):
    ax = axes[idx]
    traj = r["fade_descriptors"]["trajectory"]
    months = [p["month"] for p in traj]
    values = [p["relative_value"] for p in traj]
    color = colors.get(r["synthetic_label"], "#3498db")
    
    ax.plot(months, values, color=color, linewidth=2, marker='o', markersize=4)
    ax.fill_between(months, values, alpha=0.2, color=color)
    ax.set_title(f"{r['repo'][:30]}\n({r['synthetic_label']})", fontsize=10)
    ax.set_xlabel("Month")
    ax.set_ylabel("Relative Involvement")
    ax.set_ylim(0, 1.5)
    ax.grid(alpha=0.3)

    # Add fade index annotation
    ax.text(0.05, 0.95, f"Fade: {r['fade_descriptors']['fade_index']:.3f}",
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle("Founder Fade Trajectories by Synthetic Label", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("fade_trajectories.png", dpi=150, bbox_inches='tight')
plt.show()

In [16]:
# ── Feature importance bar chart ──
fig, ax = plt.subplots(figsize=(12, 8))

feat_names = list(perm_imp.keys())
feat_values = list(perm_imp.values())

# Sort by importance
sorted_idx = np.argsort(feat_values)[::-1]
feat_names = [feat_names[i] for i in sorted_idx]
feat_values = [feat_values[i] for i in sorted_idx]

colors_bar = ['#e74c3c' if v > 0 else '#3498db' for v in feat_values]
bars = ax.barh(feat_names, feat_values, color=colors_bar, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel("AUC Drop (Permutation Importance)")
ax.set_title("Permutation Feature Importance\n(Positive = important for prediction)", fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

In [17]:
# ── Model performance summary ──
print("\n" + "=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"\nLogistic Regression (LOOCV):")
print(f"  Static features AUC:    {log_results.get('static_auc_mean', 'N/A')}")
print(f"  Trajectory features AUC: {log_results.get('trajectory_auc_mean', 'N/A')}")
print(f"  Combined AUC:           {log_results.get('combined_auc_mean', 'N/A')}")
print(f"  Bootstrap mean:         {log_results.get('combined_auc_bootstrap_mean', 'N/A')}")
print(f"  95% CI: [{log_results.get('combined_auc_ci_95_low', 'N/A')}, {log_results.get('combined_auc_ci_95_high', 'N/A')}]")
print(f"  Projects evaluated:     {log_results.get('n_projects', 'N/A')}")

print(f"\nCox Proportional Hazards:")
print(f"  Concordance index:      {cox_results.get('concordance_index', 'N/A')}")
if 'error' in cox_results:
    print(f"  Error: {cox_results['error']}")

print(f"\nLabel Distribution:")
print(f"  ACTIVE (SURVIVE):       {survive_count}")
print(f"  INACTIVE (COLLAPSE):    {collapse_count}")

print(f"\nTop 5 Important Features:")
sorted_imp = sorted(perm_imp.items(), key=lambda x: x[1], reverse=True)
for name, val in sorted_imp[:5]:
    print(f"  {name:25s}: {val:.6f}")


MODEL PERFORMANCE SUMMARY

Logistic Regression (LOOCV):
  Static features AUC:    nan
  Trajectory features AUC: nan
  Combined AUC:           nan
  Bootstrap mean:         nan
  95% CI: [nan, nan]
  Projects evaluated:     9

Cox Proportional Hazards:
  Concordance index:      nan
  Error: Convergence halted due to matrix inversion problems. Suspicion is high collinearity. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-modelMatrix is singular.

Label Distribution:
  ACTIVE (SURVIVE):       5
  INACTIVE (COLLAPSE):    5

Top 5 Important Features:
  slope                    : 0.000000
  convexity                : 0.000000
  time_to_onset            : 0.000000
  cliff                    : 0.000000
  plateau                  : 0.000000
